# D0: трудные негативы и оценки учителя

> Парная к [`kaggle_phase_d.ipynb`](kaggle_phase_d.ipynb), которая на весь
> этап D и рассчитана на работу руками. Здесь только D0 — самый дорогой шаг
> этапа и единственный, чей результат нужен сразу двум этапам рецепта
> (трудные негативы этапу B, оценки учителя этапу C). Поэтому его считают
> отдельно: рецепт можно переделывать сколько угодно, не повторяя
> подготовку.

Тетрадь рассчитана **только** на **Save Version → Save & Run All (Commit)**.
Руками по ячейкам её гонять не надо: commit стартует в чистой сессии, делает
всё сам и **сохраняет `/kaggle/working` как вывод версии**, который дальше
подключается входом следующей тетради.

**Перед запуском, один раз:**

1. **+ Add Input** → датасеты `ru-fin-retrieval`, `ru-fin-queries`
   и модели `intfloat/multilingual-e5-small` и `BAAI/bge-reranker-v2-m3`;
2. **Settings → Accelerator → GPU T4 ×2.** Обе карты заняты делом: оценки
   учителя считаются через `DataParallel`, работа делится по парам, связи
   между примерами нет. Просить две карты и использовать одну — худший
   вариант, квота идёт за обе;
3. интернет включён;
4. **Save Version → Save & Run All (Commit)**.

**Веса берутся из моделей, подключённых входом, а не качаются.** Это
быстрее и не зависит от сети, но зеркало — чужая копия, и подменённые веса
ничем себя не выдадут: дистилляция пойдёт, лосс упадёт, а учить будут
не тому. Поэтому `gpu_prepare.py` сверяет устройство модели с ожидаемым
и воспроизводит уже посчитанную выдачу `base__hybrid-rerank.jsonl`:
совпали первые места — тот самый учитель, которым измерено 0,690.

**Бюджет.** Проба, затем 183 тысячи проходов кросс-энкодера на двух
картах. Час-полтора.

**Чего тут нет нарочно:** нулевой точки, обучения и всего рецепта. Они
считаются отдельно и в подготовке не нуждаются.

In [ ]:
# ──────── НАСТРОЙКИ ПРОГОНА ────────
# Меняются только здесь.

ГЛУБИНА = 30     # кандидатов на вопрос: 6112 × 30 = 183 тысячи проходов учителя
НЕГАТИВОВ = 2    # сколько негативов на вопрос нужно этапу B
# Проба выключена: она кодирует те же сорок тысяч обучающих
# фрагментов, что и полный прогон, то есть удваивает самый
# дорогой шаг. Полный прогон теперь и сам отчитывается блоками,
# и пишет результат по пачкам — проверять путь отдельно незачем.
ПРОБА = False
ЗАМЕР = 2048     # на скольких парах мерить скорость учителя
ПРЕДЕЛ_ЧАСОВ = 6 # по замеру не укладываемся — прогон откажется начинаться
БАТЧ = 64        # пар кросс-энкодера за один проход

import time
НАЧАЛО = time.time()


def прошло() -> str:
    с = time.time() - НАЧАЛО
    return f'{с // 3600:.0f} ч {(с % 3600) // 60:02.0f} мин'

## 1. Карта и токенПадать надо здесь, на первой секунде, а не через сорок минут на загрузкевесов. Токен не обязателен — без него скачивание режется по скорости, —но знать, есть он или нет, надо до прогона, а не после.

In [ ]:
import os

!nvidia-smi --query-gpu=name,memory.total --format=csv

ТОКЕН = False
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token and len(token) > 10:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = token
        ТОКЕН = True
        print('токен HuggingFace на месте, длина', len(token))
except Exception as e:
    print('токена нет:', e)
if not ТОКЕН:
    print('Веса скачаются и без него, но медленнее. Add-ons -> Secrets -> HF_TOKEN')

import torch
print('карта:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ')
print('карт видно:', torch.cuda.device_count(),
      '- если 2, вторая простаивает, а квота идёт за обе')

## 2. Библиотеки: ничего не ставимD0 не обучает, а считает: нужны `transformers`, `sentence-transformers`и `torch`, и все три есть в образе Kaggle. Установка `-U` здесь стоила быпяти-пятнадцати минут квоты и добавила бы риск перебора версий, которыйкрутится часами. Версии просто печатаются: если что-то не сойдётся,это будет видно сразу, а не через полтора часа.Обучению на следующем шаге нужен `sentence-transformers` не ниже 3.0 —но обучение здесь и не запускается.

In [ ]:
import sentence_transformers, transformers
print('transformers', transformers.__version__)
print('sentence-transformers', sentence_transformers.__version__)
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sentence_transformers import SentenceTransformer
print('всё, что нужно D0, на месте')

## 3. Код и данные на место

In [ ]:
import glob, os, shutil, zipfile


def find_source(root='/kaggle/input', max_depth=4):
    found, archives = [], []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs:
            stale = ('__notebook__.ipynb' in files or 'embeddings' in dirs
                     or '__output__.json' in files)
            found.append((1 if stale else 0, cur))
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    if found:
        found.sort()
        return found[0][1], None
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


src, archive = find_source()
if src is None:
    for cur, dirs, files in os.walk('/kaggle/input'):
        depth = cur.count('/') - '/kaggle/input'.count('/')
        if depth >= 3:
            dirs.clear()
        print('   ' * depth + os.path.basename(cur) + '/')
        for f in sorted(files)[:6]:
            print('   ' * (depth + 1) + f)
    raise SystemExit('датасет не подключён: + Add Input -> ru-fin-retrieval')

if archive:
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for имя in os.listdir(src):
        путь, цель = os.path.join(src, имя), os.path.join('/kaggle/working', имя)
        if os.path.isdir(путь):
            shutil.copytree(путь, цель, dirs_exist_ok=True)
        elif имя.endswith(('.py', '.md', '.gz')):
            shutil.copyfile(путь, цель)

# D0 нужны ровно три файла помимо корпуса, и каждый по своей причине.
нехватка = []
for имя, зачем in (('acts.jsonl', 'корпус, из него нарезка'),
                   ('split.json', 'кандидаты только из обучающих актов'),
                   ('synthetic_train.jsonl', 'сами обучающие вопросы'),
                   ('chunks_base.jsonl', 'каноническая нарезка'),
                   ('queries.jsonl', 'сверка учителя с этапом 2'),
                   ('base__hybrid.jsonl', 'кандидаты для сверки'),
                   ('base__hybrid-rerank.jsonl', 'эталон порядка для сверки')):
    нашлось = (glob.glob(f'/kaggle/input/**/{имя}', recursive=True)
               + glob.glob(f'/kaggle/input/**/{имя}.gz', recursive=True))
    print(f'{имя:<24} {"есть" if нашлось else "НЕТ":<5} {зачем}')
    if not нашлось:
        нехватка.append(имя)
assert not нехватка, f'не подключены входы с файлами: {нехватка}'
print('\nпрошло', прошло())

### 3б. Каноническая нарезка. Без неё прогона не будетИдентификатор фрагмента позиционный (`акт#номер`), поэтому при сдвигеграниц нарезки он не исчезает, а начинает указывать на другой текст:эталон обучающей выборки съехал бы молча. Ячейка должна напечатать**62594**, и `gpu_prepare.py` сверит это число со статистикой сплитаещё раз, уже сам.

In [ ]:
import glob, gzip, os, shutil

dst = '/kaggle/working/chunks/base.jsonl'
if os.path.exists(dst):
    os.remove(dst)
найдено = (glob.glob('/kaggle/input/**/chunks_base.jsonl', recursive=True)
           + glob.glob('/kaggle/input/**/chunks_base.jsonl.gz', recursive=True))
src = найдено[0]
os.makedirs('/kaggle/working/chunks', exist_ok=True)
opener = gzip.open if src.endswith('.gz') else open
with opener(src, 'rb') as fi, open(dst, 'wb') as fo:
    shutil.copyfileobj(fi, fo, length=1 << 20)
n = sum(1 for _ in open(dst, encoding='utf-8'))
print('источник:', src)
print('фрагментов:', n)
assert n == 62594, f'ожидалось 62594, а не {n}: нарезка не каноническая'

## 4. Проба: пятьдесят вопросов, глубина 10Десять минут вместо полутора часов неизвестности. Проверяется не качество,а путь целиком: кандидаты, оценки учителя, отсечка негативов, запись файла.Отдельно печатается, **сработала ли отсечка**: если она не выбросилани одного кандидата, значит шкала оценок определена неверно, и негативамислужили бы неразмеченные позитивы.

In [ ]:
# Запуск через subprocess, а не через «!»: подстановка переменных в шелл-магию
# IPython идёт по регулярному выражению, а имена здесь кириллические, и это
# ровно то место, где прогон на полтора часа падает на первой секунде из-за
# подстановки. Список аргументов собирается питоном и ни во что не
# разворачивается. Вывод при этом по-прежнему идёт живьём.
import subprocess, sys

ПОДГОТОВКА = ['python', 'gpu_prepare.py', '--per-query', str(НЕГАТИВОВ),
              '--probe', str(ЗАМЕР), '--max-hours', str(ПРЕДЕЛ_ЧАСОВ)]

if ПРОБА:
    subprocess.run(ПОДГОТОВКА + ['--limit', '50', '--depth', '10',
                                 '--out', '/kaggle/working/proba'],
                   cwd='/kaggle/working', check=True)

    import json
    rec = [json.loads(l) for l in open(
        '/kaggle/working/proba/train_prepared.jsonl', encoding='utf-8')]
    r = json.load(open('/kaggle/working/proba/report_prepare.json', encoding='utf-8'))
    print(f'\nзаписей {len(rec)}, кандидатов у первой {len(rec[0]["candidates"])}')
    print('шкала оценок:', r['шкала оценок'])
    print('места эталона у учителя:', [x['teacher_place'] for x in rec[:10]])
    отброшено = r['негативы']['отброшено кандидатов']
    print('отброшено кандидатов:', отброшено)
    убрала = sum(отброшено.get(k, 0) for k in
                 ('учитель считает релевантным', 'оценка близка к эталону'))
    assert убрала > 0, ('отсечка по оценке учителя не убрала ни одного кандидата: '
                        'скорее всего шкала оценок определена неверно, и негативами '
                        'служили бы неразмеченные позитивы')
    print(f'отсечка сработала: убрано {убрала}')
    print('\nпрошло', прошло())
else:
    print('проба выключена')

## 5. Полный прогон6112 вопросов, глубина 30, то есть 183 тысячи проходов кросс-энкодера.Полтора часа. Скрипт печатает скорость и оставшееся время каждые стобатчей — по ним видно, укладываемся ли, задолго до конца.Шаг проверяет, нет ли готового файла, и пропускает себя: при повторномпрогоне полтора часа не тратятся заново.

In [ ]:
subprocess.run(ПОДГОТОВКА + ['--depth', str(ГЛУБИНА),
                             '--batch-size', str(БАТЧ),
                             '--out', '/kaggle/working/train'],
               cwd='/kaggle/working', check=True)

print('\nпрошло', прошло())

## 6. Что получилось

In [ ]:
import json, os

путь = '/kaggle/working/train/report_prepare.json'
assert os.path.exists(путь), 'отчёта нет: полный прогон не дошёл до конца'
r = json.load(open(путь, encoding='utf-8'))
for k, v in r.items():
    if isinstance(v, dict):
        print(f'{k}:')
        for k2, v2 in v.items():
            if isinstance(v2, dict):
                print(f'   {k2}:')
                for k3, v3 in v2.items():
                    print(f'      {k3:<34} {v3}')
            else:
                print(f'   {k2:<34} {v2}')
    else:
        print(f'{k:<34} {v}')

файл = '/kaggle/working/train/train_prepared.jsonl'
print(f'\nфайл: {os.path.getsize(файл) / 1048576:.1f} МБ')
print('прошло всего', прошло())

# Проба в вывод версии не едет: она на пятидесяти вопросах и годится только
# как проверка пути, а спутать её с настоящей подготовкой очень легко —
# имена файлов одинаковые.
import shutil
shutil.rmtree('/kaggle/working/proba', ignore_errors=True)
shutil.rmtree('/kaggle/working/chunks', ignore_errors=True)
print('\nпроба и кэш нарезки убраны из вывода версии')

## 7. Что дальшеВывод этой версии сохранён автоматически — это и есть смысл commit.После обновления квоты:1. открыть [`kaggle_phase_d.ipynb`](kaggle_phase_d.ipynb);2. **+ Add Input → Your Work** → вывод этой тетради. `run_phase_d.py`   найдёт `train_prepared.jsonl` сам;3. пропустить её шаги 4 и 5 (проба и D0 — уже сделаны) и начинать   с шага 6, нулевой точки.На мак скачать `train/train_prepared.jsonl` (в `data/queries/`)и `train/report_prepare.json` (в `docs/raw/`): по ним `make check-train`проверит отсечку негативов тем же кодом, которым её считала видеокарта.